# データ収集

NPMパッケージのメタデータ、README、依存関係などを収集する。

In [ ]:
# インストール（Colab用）
!pip install -q requests beautifulsoup4 numpy pandas scikit-learn transformers sentence-transformers nltk

In [ ]:
import sys
import os
from pathlib import Path

# パス設定（Colab用）
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    
    # プロジェクトディレクトリを追加
    project_dir = Path('/content/drive/MyDrive/evil_package')
    sys.path.insert(0, str(project_dir))
else:
    project_dir = Path('..')

os.chdir(project_dir)

In [ ]:
from src.data.collectors.npm_collector import NPMCollector
from src.data.collectors.osv_collector import OSVCollector
from src.data.pipeline import DataPipeline
import json
import pandas as pd

## 悪性パッケージデータの収集

既存研究のデータセットやOSVから悪性パッケージのリストを取得する。

In [ ]:
# 悪性パッケージリスト（例：既存研究から取得、または手動でリスト化）
# 実際の研究では、CerebroやMalGuardのデータセットから取得
malicious_packages = [
    # ここに悪性パッケージ名を追加
    # 例: "malicious-package-1", "typosquatting-package"
]

## 良性パッケージデータの収集

NPMレジストリから人気パッケージやランダムサンプルを取得する。

In [ ]:
# 人気パッケージの取得（NPM検索API使用）
collector = NPMCollector()

# 人気パッケージを検索
popular_searches = ["react", "vue", "angular", "express", "lodash", "axios"]
popular_packages = []

for query in popular_searches:
    results = collector.search_packages(query, size=10)
    for result in results:
        package_name = result.get('package', {}).get('name')
        if package_name:
            popular_packages.append(package_name)

popular_packages = list(set(popular_packages))  # 重複除去
print(f"Found {len(popular_packages)} popular packages")

In [ ]:
# データ収集パイプラインの実行
pipeline = DataPipeline(output_dir="data/processed", popular_packages=popular_packages)

# 悪性パッケージの処理
malicious_labels = [1] * len(malicious_packages)
malicious_results = pipeline.process_package_list(malicious_packages, labels=malicious_labels)

# 良性パッケージの処理
benign_labels = [0] * len(popular_packages)
benign_results = pipeline.process_package_list(popular_packages, labels=benign_labels)

print(f"Processed {len(malicious_results)} malicious packages")
print(f"Processed {len(benign_results)} benign packages")

## 時系列分割

データを時系列で分割する（概念ドリフト評価用）。

In [ ]:
# 全データを結合
all_results = malicious_results + benign_results

# 時系列分割
splits = pipeline.create_time_split(
    all_results,
    train_end="2023-12-31",
    val_end="2024-06-30"
)

print(f"Train: {len(splits['train'])} samples")
print(f"Val: {len(splits['val'])} samples")
print(f"Test: {len(splits['test'])} samples")

# 分割データを保存
with open("data/splits/train.json", "w") as f:
    json.dump(splits['train'], f, indent=2, default=str)

with open("data/splits/val.json", "w") as f:
    json.dump(splits['val'], f, indent=2, default=str)

with open("data/splits/test.json", "w") as f:
    json.dump(splits['test'], f, indent=2, default=str)

## データの確認

収集したデータの統計を確認する。

In [ ]:
# データの基本統計
df = pd.DataFrame([
    {
        'package_name': r['package_name'],
        'label': r['label'],
        'created': r['metadata'].get('created', ''),
        'version_count': len(r['metadata'].get('versions', [])),
        'dependency_count': len(r['metadata'].get('dependencies', {})),
    }
    for r in all_results if r
])

print(df.describe())
print("\nLabel distribution:")
print(df['label'].value_counts())